In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from tqdm import tqdm
import mygene

In [ ]:
bulk = pd.read_csv("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/mosaic_dataset/RNAseq/counts/raw_counts.tsv", sep="\t")

In [ ]:
bulk

In [ ]:
mg = mygene.MyGeneInfo()
ensembl_ids = bulk['gene'].values.tolist()

# Get symbol + biotype
out = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields="symbol,name,type_of_gene",
    species="human",
)
map_df = pd.DataFrame(out)

In [ ]:
# Clean up the mygene output
map_df = map_df.drop_duplicates(subset=['query']) # Handle mygene query duplicates

In [ ]:
# List of gene biotypes we want to exclude from aggregation
biotypes_to_exclude = [
    'snRNA',        # Small nuclear RNA
    'pseudo',       # Pseudogenes (non-functional gene copies)
    'snoRNA',       # Small nucleolar RNA
    'rRNA',         # Ribosomal RNA
    'scRNA',        # Small cytoplasmic RNA
    'tRNA'          # Transfer RNA
]

# Keep only the genes that are NOT in the exclusion list
map_df_filtered = map_df[~map_df['type_of_gene'].isin(biotypes_to_exclude)]

print(f"Original mappings: {len(map_df)}")
print(f"Mappings after filtering: {len(map_df_filtered)}")

In [ ]:
map_df_filtered

In [ ]:
map_df_filtered_filtered = map_df_filtered[map_df_filtered["name"].str.contains("pseudogene")==False]

In [ ]:
# Create the mapping from the CLEANED dataframe
mapping = map_df_filtered_filtered.set_index('query')['symbol']

# Add the gene symbols as a new column
bulk['symbol'] = bulk['gene'].map(mapping)

# Fill unmapped genes with their original Ensembl ID
bulk['symbol'].fillna(bulk['gene'], inplace=True)

# Group by symbol and sum the expression data
numeric_cols = bulk.select_dtypes(include=np.number).columns
bulk_aggregated = bulk.groupby('symbol')[numeric_cols].sum().reset_index()

# Rename for consistency
bulk_final = bulk_aggregated.rename(columns={'symbol': 'gene'})

# remove all ensembl genes
bulk_final = bulk_final[~bulk_final["gene"].str.startswith("ENSG")]

print("\nFinal Aggregated DataFrame (Y_RNA and others excluded):")
bulk_final

In [ ]:
# for all gene version
#df = bulk
df = bulk_final

# remove genes expressed in less than 50% of the samples
sample_cols = df.columns[1:]
min_samples = len(sample_cols) * 0.5
df = df[df[sample_cols].gt(0).sum(axis=1) >= min_samples]

df = df.set_index('gene')

def rename_sample(col):
    new_name = col.replace('_', '')  # remove underscores
    if new_name.endswith('mRNA'):      # remove trailing 'mRNA'
        new_name = new_name[:-4]
    return new_name

# mapping for the sample columns and rename them
new_sample_names = {col: rename_sample(col) for col in sample_cols}
df_final = df.rename(columns=new_sample_names)

In [ ]:
df_final.index.name = None
dft = df_final.T
dft["sample_id"] = dft.index
dft.reset_index(drop=True, inplace=True)

In [ ]:
dft.to_csv("mofa/mofa_workflow/input_data_mosaic/bulk_05symbols.csv", index=False) # for DEGs in patient strat in downstream notebook

In [ ]:
# for hvgs version
#df = bulk
df = bulk_final

# Remove genes expressed in less than 50% of the samples
sample_cols = df.columns[1:]
min_samples = len(sample_cols) * 0.5
df = df[df[sample_cols].gt(0).sum(axis=1) >= min_samples]

# Set gene column as index for easier data manipulation ---
df = df.set_index('gene')

# Log-transform the data 
log_df = np.log2(df[sample_cols] + 1)


# Seurat-Style Highly Variable Gene (HVG) Selection

# Calculate gene-wise mean and variance on the log-transformed data
gene_stats = pd.DataFrame({
    'mean': log_df.mean(axis=1),
    'variance': log_df.var(axis=1)
})

# Create expression bins
gene_stats['mean_bin'] = pd.qcut(gene_stats['mean'], q=20, labels=False, duplicates='drop')

# For each bin, calculate the Z-score for variance
bin_groups = gene_stats.groupby('mean_bin')['variance']
bin_mean = bin_groups.transform('mean')
bin_std = bin_groups.transform('std')

# Calculate a Z-score for each gene's variance within its bin
gene_stats['variance_zscore'] = (gene_stats['variance'] - bin_mean) / bin_std

# Replace any potential NaN values (from bins with a single gene) with 0
gene_stats.fillna({'variance_zscore': 0}, inplace=True)


# Select the top N genes based on the normalized variance Z-score
num_hvgs = 5000
top_hvgs = gene_stats.nlargest(num_hvgs, 'variance_zscore').index

print(f"Original number of genes: {len(df)}")
print(f"Selected top {num_hvgs} highly variable genes.")

# Filter original (non-log) dataframe to keep only these HVGs
df_hvg = df.loc[top_hvgs]

# Create a mapping for the sample columns and rename them
new_sample_names = {col: rename_sample(col) for col in sample_cols}
df_hvg = df_hvg.rename(columns=new_sample_names)

In [ ]:
df_hvg

In [ ]:
df_hvg.index.name = None

In [ ]:
dft = df_hvg.T

In [ ]:
dft["sample_id"] = dft.index

In [ ]:
dft.reset_index(drop=True, inplace=True)

In [ ]:
dft

In [ ]:
#dft.to_csv("mofa/resources/bulk_5000hvg_counts.csv", index=False)
dft.to_csv("mofa/resources/bulk_5000hvg_counts_symbols.csv", index=False) 

In [ ]:
dft.to_csv("mofa/mofa_workflow/input_data_mosaic/bulk_5000hvg_counts_symbols.csv", index=False) # input for mofa model